# Full Dataset Cleaning & Combination

Goals:

- Create a reusable cleaning pipeline
- Apply cleaning to all 8 datasets
- Standardize attack labels
- Merge everything into one master dataframe
- Validate the final dataset

In [12]:
from pathlib import Path

import pandas as pd
import numpy as np

In [13]:
DATA_DIR = Path("../data/raw")

csv_files = sorted(DATA_DIR.glob("*.csv"))

print(f"Found {len(csv_files)} files:\n")

for file in csv_files:
    print(file.name)

Found 8 files:

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [14]:
def clean_dataframe(df):

    df.columns = df.columns.str.strip()

    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    label_mapping = {
        "Web Attack � Brute Force":
            "Web Attack - Brute Force",

        "Web Attack � XSS":
            "Web Attack - XSS",

        "Web Attack � Sql Injection":
            "Web Attack - SQL Injection"
    }

    if "Label" in df.columns:
        df["Label"] = df["Label"].replace(label_mapping)

    return df

In [15]:
daily_dfs = {}
cleaned_dfs = []

for file in csv_files:

    print(f"Processing {file.name}")

    df = pd.read_csv(file)

    df = clean_dataframe(df)

    day_name = file.name.split(".")[0]
    df["Day"] = day_name

    daily_dfs[day_name] = df
    cleaned_dfs.append(df)

print("\nDone!")

Processing Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Processing Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Processing Friday-WorkingHours-Morning.pcap_ISCX.csv
Processing Monday-WorkingHours.pcap_ISCX.csv
Processing Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Processing Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Processing Tuesday-WorkingHours.pcap_ISCX.csv
Processing Wednesday-workingHours.pcap_ISCX.csv

Done!


In [16]:
master_df = pd.concat(
    cleaned_dfs,
    ignore_index=True
)

In [19]:
print(master_df.shape)

master_df[["Day", "Label"]].head()

(2830743, 80)


,Day,Label
0,Friday-WorkingHours-Afternoon-DDos,BENIGN
1,Friday-WorkingHours-Afternoon-DDos,BENIGN
2,Friday-WorkingHours-Afternoon-DDos,BENIGN
3,Friday-WorkingHours-Afternoon-DDos,BENIGN
4,Friday-WorkingHours-Afternoon-DDos,BENIGN


In [20]:
sorted(master_df["Label"].unique())

['BENIGN',
 'Bot',
 'DDoS',
 'DoS GoldenEye',
 'DoS Hulk',
 'DoS Slowhttptest',
 'DoS slowloris',
 'FTP-Patator',
 'Heartbleed',
 'Infiltration',
 'PortScan',
 'SSH-Patator',
 'Web Attack - Brute Force',
 'Web Attack - SQL Injection',
 'Web Attack - XSS']

In [21]:
memory_gb = (
    master_df.memory_usage(deep=True).sum()
    / 1024**3
)

print(f"Memory usage: {memory_gb:.2f} GB")

Memory usage: 1.99 GB


In [22]:
master_df.groupby("Day")["Label"].value_counts()

Day                                            Label                     
Friday-WorkingHours-Afternoon-DDos             DDoS                          128027
                                               BENIGN                         97718
Friday-WorkingHours-Afternoon-PortScan         PortScan                      158930
                                               BENIGN                        127537
Friday-WorkingHours-Morning                    BENIGN                        189067
                                               Bot                             1966
Monday-WorkingHours                            BENIGN                        529918
Thursday-WorkingHours-Afternoon-Infilteration  BENIGN                        288566
                                               Infiltration                      36
Thursday-WorkingHours-Morning-WebAttacks       BENIGN                        168186
                                               Web Attack - Brute Force        1507
  

In [23]:
master_df.to_csv(
    "../data/processed/combined_cleaned.csv",
    index=False
)

print("Saved successfully!")

Saved successfully!
